# LeetCode 3242 — Design Neighbor Sum Service bằng Python

Link đề: https://leetcode.com/problems/design-neighbor-sum-service/ · Độ khó: **Easy** · Chủ đề: Design, Array, Hash Table, Matrix, Simulation

Skeleton LeetCode (bài thiết kế lớp — LeetCode tạo đối tượng rồi gọi các phương thức theo dãy lệnh):

```python
class NeighborSum:

    def __init__(self, grid: List[List[int]]):
        ...

    def adjacentSum(self, value: int) -> int:
        ...

    def diagonalSum(self, value: int) -> int:
        ...


# Your NeighborSum object will be instantiated and called as such:
# obj = NeighborSum(grid)
# param_1 = obj.adjacentSum(value)
# param_2 = obj.diagonalSum(value)
```

Ghi chú: mọi đáp án trong notebook đã được kiểm chứng bằng code (kể cả đối chiếu hàng nghìn lưới và truy vấn ngẫu nhiên với bản vét cạn). Hai ví dụ chính lấy từ đề; những ví dụ tự dựng thêm có ghi rõ "tự dựng" — nếu muốn khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho lưới `n x n` gồm các giá trị **đôi một khác nhau** trong khoảng `[0, n² − 1]`; thiết kế lớp `NeighborSum` có `adjacentSum(value)` (tổng các ô **kề cạnh**: trên, dưới, trái, phải của ô chứa `value`) và `diagonalSum(value)` (tổng các ô **kề chéo**: bốn góc).
2. Nắm các công cụ Python: **lớp** (`class`, `__init__`, `self`, thuộc tính), **chỉ số 2 chiều** `grid[i][j]`, `enumerate`, **`dict` giá trị → tọa độ**, **bảng độ lệch hướng** `(di, dj)`, kiểm tra **biên** `0 <= x < n`, và cách LeetCode gọi lớp bằng dãy lệnh.
3. Đi từ cách **mỗi truy vấn quét cả lưới** → **dựng sẵn `dict` giá trị → vị trí** → **tính trước mọi tổng ngay trong `__init__`** → **thêm viền số `0` để bỏ kiểm tra biên** → **hàm sinh láng giềng dùng chung**.
4. Hiểu mấu chốt của bài thiết kế: phân chia việc **một lần ở `__init__`** (chuẩn bị dữ liệu) với việc **nhiều lần ở truy vấn** (tra cứu nhanh) — vì giá trị đôi một khác nhau nên mỗi giá trị xác định **đúng một ô**, và `dict` cho phép tìm ô đó trong `O(1)`.
5. Tránh các lỗi kinh điển: quên `self.` (dữ liệu biến mất sau `__init__`), **chỉ số âm của Python quay vòng** khiến sai thầm lặng khi quên kiểm tra biên, nhầm **giá trị** với **chỉ số hàng**, và nhầm bộ hướng **kề cạnh** với **kề chéo** (hoặc lấy cả 8 ô).

## 2. Đề bài & ví dụ

Cho mảng 2 chiều `grid` cỡ `n x n` gồm các phần tử **đôi một khác nhau** trong khoảng `[0, n² − 1]`.

Hãy cài đặt lớp `NeighborSum`:

- `NeighborSum(int[][] grid)` khởi tạo đối tượng.
- `int adjacentSum(int value)` trả về **tổng** các phần tử là **láng giềng kề cạnh** của `value` trong `grid`, tức nằm ở **trên, trái, phải hoặc dưới** ô chứa `value`.
- `int diagonalSum(int value)` trả về **tổng** các phần tử là **láng giềng chéo** của `value`, tức nằm ở **trên-trái, trên-phải, dưới-trái hoặc dưới-phải** ô chứa `value`.

**Ví dụ 1**

- Input (dãy lệnh): `["NeighborSum", "adjacentSum", "adjacentSum", "diagonalSum", "diagonalSum"]`, tham số: `[[[[0, 1, 2], [3, 4, 5], [6, 7, 8]]], [1], [4], [4], [8]]`
- Output: `[null, 6, 16, 16, 4]`
- Giải thích: láng giềng kề cạnh của `1` là `0, 2, 4` (tổng `6`); của `4` là `1, 3, 5, 7` (tổng `16`); láng giềng chéo của `4` là `0, 2, 6, 8` (tổng `16`); láng giềng chéo của `8` chỉ có `4`.

**Ví dụ 2**

- Input: `["NeighborSum", "adjacentSum", "diagonalSum"]`, tham số: `[[[[1, 2, 0, 3], [4, 7, 15, 6], [8, 9, 10, 11], [12, 13, 14, 5]]], [15], [9]]`
- Output: `[null, 23, 45]`
- Giải thích: láng giềng kề cạnh của `15` là `0, 10, 7, 6` (tổng `23`); láng giềng chéo của `9` là `4, 12, 14, 15` (tổng `45`).

**Ràng buộc:** `3 <= n == grid.length == grid[0].length <= 10`; `0 <= grid[i][j] <= n² − 1`; mọi `grid[i][j]` đôi một khác nhau (nên lưới là một **hoán vị** của `0..n²−1`); `value` ở cả hai hàm nằm trong `[0, n² − 1]`; tối đa `2 * n²` lời gọi `adjacentSum` / `diagonalSum`.

In [1]:
grid1 = [[0, 1, 2], [3, 4, 5], [6, 7, 8]]
grid2 = [[1, 2, 0, 3], [4, 7, 15, 6], [8, 9, 10, 11], [12, 13, 14, 5]]

# (lưới, danh sách (tên hàm, tham số), kết quả kỳ vọng — null của hàm tạo là None)
examples = [
    (grid1, [("adjacentSum", 1), ("adjacentSum", 4), ("diagonalSum", 4), ("diagonalSum", 8)], [6, 16, 16, 4]),
    (grid2, [("adjacentSum", 15), ("diagonalSum", 9)], [23, 45]),
]
for grid, queries, expected in examples:
    print(f"lưới {len(grid)}x{len(grid)}, truy vấn {queries} -> kỳ vọng {expected}")

lưới 3x3, truy vấn [('adjacentSum', 1), ('adjacentSum', 4), ('diagonalSum', 4), ('diagonalSum', 8)] -> kỳ vọng [6, 16, 16, 4]
lưới 4x4, truy vấn [('adjacentSum', 15), ('diagonalSum', 9)] -> kỳ vọng [23, 45]


## 3. Kiến thức nền

### 3.1. Lớp, `__init__`, `self` và thuộc tính

Một **lớp** (`class`) gói **dữ liệu** và **hàm** (phương thức) đi cùng nhau. `__init__` chạy **một lần** khi tạo đối tượng (`obj = NeighborSum(grid)`). `self` là chính đối tượng đó; gán `self.x = ...` để **lưu dữ liệu lâu dài** cho các phương thức khác. Biến thường (không có `self.`) trong `__init__` **mất đi** khi hàm kết thúc.

In [2]:
class Counter3:
    def __init__(self, start):
        self.value = start            # thuộc tính: sống cùng đối tượng
        tmp = start * 10              # biến cục bộ: mất sau __init__

    def add(self, k):
        self.value += k
        return self.value


c = Counter3(5)
print(c.add(2), c.add(3))             # 7 10
print(hasattr(c, "value"), hasattr(c, "tmp"))   # True False

7 10
True False


### 3.2. LeetCode gọi lớp thế nào: dãy lệnh `ops` và `args`

Với bài thiết kế, LeetCode đưa hai danh sách song song: `ops` (tên lệnh) và `args` (tham số). Phần tử đầu là **hàm tạo**; mỗi lệnh sau là một **phương thức** gọi trên cùng đối tượng. Kết quả của hàm tạo hiển thị là `null` (Python `None`). Ta mô phỏng bằng hàm `run_ops` — dùng `getattr(obj, tên)(*tham_số)` để gọi phương thức theo tên.

In [3]:
def run_ops(cls, ops, args):
    """Mô phỏng LeetCode: ops[0] là tên lớp, args[0] là tham số hàm tạo."""
    obj = cls(*args[0])
    out = [None]
    for name, a in zip(ops[1:], args[1:]):
        out.append(getattr(obj, name)(*a))
    return out


class Demo:
    def __init__(self, base):
        self.base = base
    def plus(self, x):
        return self.base + x
    def times(self, x):
        return self.base * x


print(run_ops(Demo, ["Demo", "plus", "times"], [[10], [1], [3]]))     # [None, 11, 30]

[None, 11, 30]


### 3.3. Chỉ số 2 chiều `grid[i][j]` và `enumerate`

`grid[i]` là **hàng thứ `i`** (đếm từ `0` từ trên xuống), `grid[i][j]` là ô ở **hàng `i`, cột `j`**. Để duyệt kèm chỉ số: `for i, row in enumerate(grid): for j, x in enumerate(row): ...` — `i` là chỉ số hàng, `j` chỉ số cột, `x` là giá trị.

In [4]:
grid = [[0, 1, 2], [3, 4, 5], [6, 7, 8]]
print(grid[1][2])                                   # 5 (hàng 1, cột 2)
for i, row in enumerate(grid):
    for j, x in enumerate(row):
        if x in (0, 4, 8):
            print(f"giá trị {x} ở ({i}, {j})")

5
giá trị 0 ở (0, 0)
giá trị 4 ở (1, 1)
giá trị 8 ở (2, 2)


### 3.3b. `dict`: giá trị → tọa độ

Các giá trị **đôi một khác nhau** nên có thể lập `pos = {giá_trị: (hàng, cột)}`. Khi đó `pos[value]` cho tọa độ ô trong `O(1)` thay vì phải quét cả lưới. Tọa độ lưu dưới dạng **tuple** `(i, j)` và có thể **bóc tách** `i, j = pos[value]`.

In [5]:
pos = {}
for i, row in enumerate(grid):
    for j, x in enumerate(row):
        pos[x] = (i, j)
print(pos)                    # {0: (0, 0), 1: (0, 1), ..., 8: (2, 2)}
i, j = pos[4]
print(i, j)                   # 1 1

{0: (0, 0), 1: (0, 1), 2: (0, 2), 3: (1, 0), 4: (1, 1), 5: (1, 2), 6: (2, 0), 7: (2, 1), 8: (2, 2)}
1 1


### 3.4. Bảng độ lệch hướng `(di, dj)` và kiểm tra biên

Láng giềng của ô `(i, j)` là các ô `(i + di, j + dj)` với `(di, dj)` lấy từ một bảng:

- **Kề cạnh:** `(-1, 0)` trên, `(1, 0)` dưới, `(0, -1)` trái, `(0, 1)` phải.
- **Kề chéo:** `(-1, -1)`, `(-1, 1)`, `(1, -1)`, `(1, 1)`.

Ô mới chỉ hợp lệ nếu `0 <= x < n` và `0 <= y < n` (kiểm tra **biên** của lưới).

In [6]:
ADJ = ((-1, 0), (1, 0), (0, -1), (0, 1))
DIAG = ((-1, -1), (-1, 1), (1, -1), (1, 1))
n = 3
i, j = 0, 1                               # ô chứa giá trị 1
for name, dirs in (("kề cạnh", ADJ), ("kề chéo", DIAG)):
    cells = []
    for di, dj in dirs:
        x, y = i + di, j + dj
        if 0 <= x < n and 0 <= y < n:     # chỉ giữ ô nằm trong lưới
            cells.append(grid[x][y])
    print(name, "của 1:", cells, "tổng", sum(cells))

kề cạnh của 1: [4, 0, 2] tổng 6
kề chéo của 1: [3, 5] tổng 8


### 3.5. Cạm bẫy: chỉ số âm của Python quay vòng

Trong Python, `lst[-1]` là **phần tử cuối**, không báo lỗi. Do đó nếu quên kiểm tra biên và tính `grid[i - 1][j]` khi `i = 0` ta được `grid[-1][j]` — **hàng cuối** — và **không có lỗi** nào được báo, kết quả chỉ **sai thầm lặng**. (Chỉ số **quá lớn** như `grid[n][j]` thì báo `IndexError`.)

In [7]:
print(grid[-1])                    # [6, 7, 8]  (hàng cuối, không lỗi!)
print(grid[0 - 1][1])              # 7: tưởng là "ô phía trên ô (0,1)" nhưng thực ra là hàng cuối
try:
    print(grid[3][0])
except IndexError as e:
    print("IndexError:", e)        # chỉ số quá lớn mới báo lỗi

[6, 7, 8]
7
IndexError: list index out of range


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lưới ví dụ 1:

```
 0  1  2
 3  4  5
 6  7  8
```

Truy vấn `adjacentSum(1)`: tìm `1` ở hàng `0`, cột `1`. Bốn hướng:

| hướng | ô | trong lưới? | giá trị |
|---|---|---|---|
| trên `(-1, 0)` | `(-1, 1)` | không (hàng `-1`) | — |
| dưới `(1, 0)` | `(1, 1)` | có | 4 |
| trái `(0, -1)` | `(0, 0)` | có | 0 |
| phải `(0, 1)` | `(0, 2)` | có | 2 |

Tổng `4 + 0 + 2 = 6` ✓. Với `diagonalSum(8)`: `8` ở `(2, 2)`; chỉ `(1, 1)` nằm trong lưới ⇒ tổng `4` ✓ (ba ô còn lại ra ngoài biên).

Trực giác cho thiết kế: **việc nặng** (tìm ô của `value`) có thể làm **sẵn một lần** ở `__init__`; mỗi truy vấn chỉ còn **cộng tối đa 4 số**.

In [8]:
grid = [[0, 1, 2], [3, 4, 5], [6, 7, 8]]
n = len(grid)
i, j = 0, 1                                        # vị trí của giá trị 1
total = 0
for name, (di, dj) in zip(("trên", "dưới", "trái", "phải"), ADJ):
    x, y = i + di, j + dj
    inside = 0 <= x < n and 0 <= y < n
    print(f"{name:>5}: ({x:>2}, {y:>2}) trong lưới? {inside}", f"giá trị {grid[x][y]}" if inside else "")
    if inside:
        total += grid[x][y]
print("adjacentSum(1) =", total)                   # 6

 trên: (-1,  1) trong lưới? False 
 dưới: ( 1,  1) trong lưới? True giá trị 4
 trái: ( 0,  0) trong lưới? True giá trị 0
 phải: ( 0,  2) trong lưới? True giá trị 2
adjacentSum(1) = 6


## 5. Cách 1 — Brute Force / baseline: mỗi truy vấn quét cả lưới

**Ý tưởng.** `__init__` chỉ **lưu** `grid`. Mỗi lần gọi `adjacentSum(value)`:

1. Quét **toàn bộ lưới** (hai vòng lặp) để tìm ô `(i, j)` chứa `value`.
2. Duyệt các hướng kề cạnh, kiểm tra biên, cộng giá trị.

`diagonalSum` giống hệt nhưng dùng bảng hướng chéo. Cách này đúng ngay theo đề, không cần cấu trúc dữ liệu phụ.

In [9]:
class NeighborSumScan:
    ADJ = ((-1, 0), (1, 0), (0, -1), (0, 1))
    DIAG = ((-1, -1), (-1, 1), (1, -1), (1, 1))

    def __init__(self, grid):
        self.grid = grid                      # chỉ lưu lưới, không chuẩn bị gì thêm

    def _find(self, value):
        for i, row in enumerate(self.grid):   # quét toàn bộ lưới mỗi lần hỏi
            for j, x in enumerate(row):
                if x == value:
                    return i, j

    def _sum(self, value, dirs):
        i, j = self._find(value)
        n = len(self.grid)
        total = 0
        for di, dj in dirs:
            x, y = i + di, j + dj
            if 0 <= x < n and 0 <= y < n:
                total += self.grid[x][y]
        return total

    def adjacentSum(self, value):
        return self._sum(value, self.ADJ)

    def diagonalSum(self, value):
        return self._sum(value, self.DIAG)


for grid, queries, expected in examples:
    got = [getattr(NeighborSumScan(grid), name)(v) for name, v in queries]
    print(got, "(kỳ vọng", expected, ")")

[6, 16, 16, 4] (kỳ vọng [6, 16, 16, 4] )
[23, 45] (kỳ vọng [23, 45] )


**Độ phức tạp (Cách 1).** Gọi `n` là cạnh lưới, `N = n²` là số ô. `__init__`: `O(1)`. Mỗi truy vấn: quét `O(N)` ô + `O(1)` cộng ⇒ **Time `O(N)` mỗi truy vấn** (`O(n²)`), **Space `O(1)` thêm**. Với `Q <= 2N` truy vấn tổng `O(N²)` ≈ `10⁴` — vẫn rất nhỏ ở `n <= 10`.

**Nút thắt:** mỗi truy vấn lại **đi tìm** ô của `value` từ đầu. Vì các giá trị đôi một khác nhau nên ta có thể **tìm sẵn một lần** cho mọi giá trị.

## 6. Ý tưởng thuật toán / bước đột phá

**Đánh chỉ mục giá trị → vị trí ở `__init__`.** Duyệt lưới một lần, ghi `pos[x] = (i, j)`. Từ đó mỗi truy vấn: tra `pos[value]` trong `O(1)`, rồi cộng tối đa `4` ô láng giềng — **`O(1)` mỗi truy vấn**.

**Dùng chung một hàm cho hai loại tổng.** `adjacentSum` và `diagonalSum` chỉ khác **bảng hướng**. Viết một hàm `_cal(value, dirs)` và truyền bảng hướng khác nhau — tránh lặp code.

**Hoặc tính sẵn mọi đáp án.** Vì chỉ có `n²` giá trị, ta có thể tính **trước** cả `adj[value]` và `diag[value]` ở `__init__`; truy vấn chỉ còn tra bảng (đổi thời gian truy vấn lấy thời gian khởi tạo).

In [10]:
pos = {}
for i, row in enumerate(grid):
    for j, x in enumerate(row):
        pos[x] = (i, j)
n = len(grid)
for value in (1, 4, 8):
    i, j = pos[value]
    adj = sum(grid[i + di][j + dj] for di, dj in ADJ if 0 <= i + di < n and 0 <= j + dj < n)
    diag = sum(grid[i + di][j + dj] for di, dj in DIAG if 0 <= i + di < n and 0 <= j + dj < n)
    print(f"value={value}: vị trí {pos[value]}, adjacentSum={adj}, diagonalSum={diag}")

value=1: vị trí (0, 0), adjacentSum=6, diagonalSum=7
value=4: vị trí (1, 0), adjacentSum=16, diagonalSum=11
value=8: vị trí (2, 0), adjacentSum=25, diagonalSum=20


## 7. Các cách tiếp theo

### Cách 2 — `dict` giá trị → vị trí + hàm dùng chung (cách của doocs)

**Giải thích từng đoạn.**

1. `__init__`: lưu `grid`, và dựng `self.pos` từ `enumerate` hai tầng.
2. `_cal(value, dirs)`: tra `pos[value]`, rồi cộng ô láng giềng hợp lệ theo bảng hướng.
3. `adjacentSum` gọi `_cal` với bảng kề cạnh, `diagonalSum` gọi với bảng kề chéo.

In [11]:
class NeighborSumDict:
    ADJ = ((-1, 0), (1, 0), (0, -1), (0, 1))
    DIAG = ((-1, -1), (-1, 1), (1, -1), (1, 1))

    def __init__(self, grid):
        self.grid = grid
        self.pos = {}                                  # giá trị -> (hàng, cột)
        for i, row in enumerate(grid):
            for j, x in enumerate(row):
                self.pos[x] = (i, j)

    def _cal(self, value, dirs):
        i, j = self.pos[value]                         # O(1)
        n = len(self.grid)
        s = 0
        for di, dj in dirs:
            x, y = i + di, j + dj
            if 0 <= x < n and 0 <= y < n:              # bỏ ô ngoài biên
                s += self.grid[x][y]
        return s

    def adjacentSum(self, value):
        return self._cal(value, self.ADJ)

    def diagonalSum(self, value):
        return self._cal(value, self.DIAG)


for grid, queries, expected in examples:
    got = [getattr(NeighborSumDict(grid), name)(v) for name, v in queries]
    print(got, "(kỳ vọng", expected, ")")

[6, 16, 16, 4] (kỳ vọng [6, 16, 16, 4] )
[23, 45] (kỳ vọng [23, 45] )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành NeighborSum khi nộp)
from typing import List


class NeighborSumDictSubmit:
    def __init__(self, grid: List[List[int]]):
        self.grid = grid
        self.d = {}
        for i, row in enumerate(grid):
            for j, x in enumerate(row):
                self.d[x] = (i, j)

    def adjacentSum(self, value: int) -> int:
        return self.cal(value, ((-1, 0), (1, 0), (0, -1), (0, 1)))

    def diagonalSum(self, value: int) -> int:
        return self.cal(value, ((-1, -1), (-1, 1), (1, -1), (1, 1)))

    def cal(self, value: int, dirs) -> int:
        i, j = self.d[value]
        n = len(self.grid)
        return sum(self.grid[i + a][j + b] for a, b in dirs if 0 <= i + a < n and 0 <= j + b < n)


print(run_ops(NeighborSumDictSubmit, ["NeighborSum", "adjacentSum", "diagonalSum"], [[grid2], [15], [9]]))     # [None, 23, 45]

[None, 23, 45]


**Độ phức tạp Cách 2:** `__init__` **`O(N)`** (`N = n²`), mỗi truy vấn **`O(1)`** (tối đa 4 láng giềng). **Space `O(N)`** cho `dict`.

### Cách 3 — Tính sẵn mọi tổng ở `__init__`

**Giải thích.** Vì mọi giá trị đều thuộc `0..N−1`, ta tạo hai danh sách `adj = [0] * N` và `diag = [0] * N`. Ở `__init__`, với **mỗi ô** `(i, j)` có giá trị `v` ta tính luôn `adj[v]` và `diag[v]`. Truy vấn chỉ là `return self.adj[value]` — **`O(1)`** thuần và không còn phải kiểm tra biên khi truy vấn. Phù hợp khi **số truy vấn lớn** (nhiều hơn số ô).

In [13]:
class NeighborSumPrecompute:
    def __init__(self, grid):
        n = len(grid)
        self.adj = [0] * (n * n)                    # adj[v] = tổng láng giềng kề cạnh của giá trị v
        self.diag = [0] * (n * n)
        adj_dirs = ((-1, 0), (1, 0), (0, -1), (0, 1))
        diag_dirs = ((-1, -1), (-1, 1), (1, -1), (1, 1))
        for i in range(n):
            for j in range(n):
                v = grid[i][j]
                for di, dj in adj_dirs:
                    x, y = i + di, j + dj
                    if 0 <= x < n and 0 <= y < n:
                        self.adj[v] += grid[x][y]
                for di, dj in diag_dirs:
                    x, y = i + di, j + dj
                    if 0 <= x < n and 0 <= y < n:
                        self.diag[v] += grid[x][y]

    def adjacentSum(self, value):
        return self.adj[value]

    def diagonalSum(self, value):
        return self.diag[value]


for grid, queries, expected in examples:
    got = [getattr(NeighborSumPrecompute(grid), name)(v) for name, v in queries]
    print(got, "(kỳ vọng", expected, ")")

[6, 16, 16, 4] (kỳ vọng [6, 16, 16, 4] )
[23, 45] (kỳ vọng [23, 45] )


**Độ phức tạp Cách 3:** `__init__` **`O(N)`** (mỗi ô xét tối đa 8 láng giềng), mỗi truy vấn **`O(1)`** chỉ là tra mảng; **Space `O(N)`** cho hai mảng.

### Cách 4 — Thêm viền số `0` quanh lưới, bỏ kiểm tra biên

**Giải thích.** Tạo lưới mới `padded` cỡ `(n + 2) × (n + 2)`: viền ngoài toàn `0`, phần giữa là `grid` (lệch `1` hàng, `1` cột). Ô láng giềng ra ngoài biên của `grid` gốc nay rơi vào viền `0`, đóng góp `0` vào tổng — nên **không cần `if` kiểm tra biên**. Vì giá trị là hoán vị `0..N−1`, ta lưu vị trí trong **mảng** `pos[value]` (thay `dict`). Lưu ý: `grid` có thể chứa giá trị `0` nhưng cộng thêm `0` không đổi tổng nên viền `0` an toàn.

In [14]:
class NeighborSumPadded:
    def __init__(self, grid):
        n = len(grid)
        self.padded = [[0] * (n + 2) for _ in range(n + 2)]      # viền 0 bao quanh
        self.pos = [None] * (n * n)                              # pos[v] = (i, j) trong lưới có viền
        for i in range(n):
            for j in range(n):
                self.padded[i + 1][j + 1] = grid[i][j]
                self.pos[grid[i][j]] = (i + 1, j + 1)

    def adjacentSum(self, value):
        i, j = self.pos[value]
        p = self.padded
        return p[i - 1][j] + p[i + 1][j] + p[i][j - 1] + p[i][j + 1]       # không cần kiểm tra biên

    def diagonalSum(self, value):
        i, j = self.pos[value]
        p = self.padded
        return p[i - 1][j - 1] + p[i - 1][j + 1] + p[i + 1][j - 1] + p[i + 1][j + 1]


for grid, queries, expected in examples:
    got = [getattr(NeighborSumPadded(grid), name)(v) for name, v in queries]
    print(got, "(kỳ vọng", expected, ")")

[6, 16, 16, 4] (kỳ vọng [6, 16, 16, 4] )
[23, 45] (kỳ vọng [23, 45] )


**Độ phức tạp Cách 4:** `__init__` **`O(N)`**, mỗi truy vấn **`O(1)`**, **Space `O(N)`** (lưới có viền cỡ `(n + 2)²`). Code truy vấn rất ngắn, không `if`, nhưng cần nhớ **dịch chỉ số `+1`**.

### Cách 5 — Hàm sinh láng giềng + `sum` biểu thức sinh

**Giải thích.** Gom việc "đi qua các láng giềng hợp lệ" vào một **generator** `_neighbors(i, j, dirs)` dùng `yield`: mỗi lần trả về giá trị của một ô láng giềng nằm trong biên. Rồi `adjacentSum = sum(self._neighbors(i, j, ADJ))`. Cách này tách rõ **"đi qua láng giềng"** khỏi **"làm gì với chúng"** (cộng, liệt kê, tìm max...) — dễ mở rộng cho biến thể.

In [15]:
class NeighborSumGenerator:
    ADJ = ((-1, 0), (1, 0), (0, -1), (0, 1))
    DIAG = ((-1, -1), (-1, 1), (1, -1), (1, 1))

    def __init__(self, grid):
        self.grid = grid
        self.pos = {x: (i, j) for i, row in enumerate(grid) for j, x in enumerate(row)}

    def _neighbors(self, value, dirs):
        i, j = self.pos[value]
        n = len(self.grid)
        for di, dj in dirs:
            x, y = i + di, j + dj
            if 0 <= x < n and 0 <= y < n:
                yield self.grid[x][y]                   # trả từng láng giềng hợp lệ

    def adjacentSum(self, value):
        return sum(self._neighbors(value, self.ADJ))

    def diagonalSum(self, value):
        return sum(self._neighbors(value, self.DIAG))


for grid, queries, expected in examples:
    got = [getattr(NeighborSumGenerator(grid), name)(v) for name, v in queries]
    print(got, "(kỳ vọng", expected, ")")
print(list(NeighborSumGenerator(grid1)._neighbors(4, NeighborSumGenerator.ADJ)))        # [1, 7, 3, 5]

[6, 16, 16, 4] (kỳ vọng [6, 16, 16, 4] )
[23, 45] (kỳ vọng [23, 45] )
[1, 7, 3, 5]


**Độ phức tạp Cách 5:** `__init__` **`O(N)`**, mỗi truy vấn **`O(1)`**, **Space `O(N)`**.

## 8. TRACE từng bước

Ta trace Cách 2 trên Ví dụ 1: dựng `pos`, rồi từng truy vấn (kèm ô nào bị loại vì ra ngoài biên); sau đó trace cả dãy lệnh kiểu LeetCode trên Ví dụ 2; và một truy vấn ở **góc** của lưới `4×4` (tự dựng) để thấy 3 hướng bị loại.

In [16]:
def trace_build(grid):
    pos = {}
    print("Dựng pos (giá trị -> (hàng, cột)):")
    for i, row in enumerate(grid):
        for j, x in enumerate(row):
            pos[x] = (i, j)
            print(f"  grid[{i}][{j}] = {x:>2}  =>  pos[{x}] = ({i}, {j})")
    return pos


def trace_query(grid, pos, value, dirs, label):
    n = len(grid)
    i, j = pos[value]
    print(f"{label}({value}): ô ({i}, {j})")
    total = 0
    for di, dj in dirs:
        x, y = i + di, j + dj
        if 0 <= x < n and 0 <= y < n:
            total += grid[x][y]
            print(f"  hướng ({di:>2},{dj:>2}) -> ô ({x}, {y}) trong lưới, giá trị {grid[x][y]}, tổng = {total}")
        else:
            print(f"  hướng ({di:>2},{dj:>2}) -> ô ({x}, {y}) NGOÀI biên, bỏ qua")
    print(f"  kết quả = {total}")
    return total


pos1 = trace_build(grid1)
assert trace_query(grid1, pos1, 1, ADJ, "adjacentSum") == 6
assert trace_query(grid1, pos1, 8, DIAG, "diagonalSum") == 4

Dựng pos (giá trị -> (hàng, cột)):
  grid[0][0] =  0  =>  pos[0] = (0, 0)
  grid[0][1] =  1  =>  pos[1] = (0, 1)
  grid[0][2] =  2  =>  pos[2] = (0, 2)
  grid[1][0] =  3  =>  pos[3] = (1, 0)
  grid[1][1] =  4  =>  pos[4] = (1, 1)
  grid[1][2] =  5  =>  pos[5] = (1, 2)
  grid[2][0] =  6  =>  pos[6] = (2, 0)
  grid[2][1] =  7  =>  pos[7] = (2, 1)
  grid[2][2] =  8  =>  pos[8] = (2, 2)
adjacentSum(1): ô (0, 1)
  hướng (-1, 0) -> ô (-1, 1) NGOÀI biên, bỏ qua
  hướng ( 1, 0) -> ô (1, 1) trong lưới, giá trị 4, tổng = 4
  hướng ( 0,-1) -> ô (0, 0) trong lưới, giá trị 0, tổng = 4
  hướng ( 0, 1) -> ô (0, 2) trong lưới, giá trị 2, tổng = 6
  kết quả = 6
diagonalSum(8): ô (2, 2)
  hướng (-1,-1) -> ô (1, 1) trong lưới, giá trị 4, tổng = 4
  hướng (-1, 1) -> ô (1, 3) NGOÀI biên, bỏ qua
  hướng ( 1,-1) -> ô (3, 1) NGOÀI biên, bỏ qua
  hướng ( 1, 1) -> ô (3, 3) NGOÀI biên, bỏ qua
  kết quả = 4


In [17]:
def trace_ops(cls, ops, args):
    obj = cls(*args[0])
    print(f"{ops[0]}(...) -> null")
    results = [None]
    for name, a in zip(ops[1:], args[1:]):
        r = getattr(obj, name)(*a)
        print(f"{name}({', '.join(map(str, a))}) -> {r}")
        results.append(r)
    return results


assert trace_ops(NeighborSumDict, ["NeighborSum", "adjacentSum", "diagonalSum"], [[grid2], [15], [9]]) == [None, 23, 45]
pos2 = {x: (i, j) for i, row in enumerate(grid2) for j, x in enumerate(row)}
assert trace_query(grid2, pos2, 1, DIAG, "diagonalSum") == 7          # ô (0,0) ở góc: chỉ còn (1,1) = 7

NeighborSum(...) -> null
adjacentSum(15) -> 23
diagonalSum(9) -> 45
diagonalSum(1): ô (0, 0)
  hướng (-1,-1) -> ô (-1, -1) NGOÀI biên, bỏ qua
  hướng (-1, 1) -> ô (-1, 1) NGOÀI biên, bỏ qua
  hướng ( 1,-1) -> ô (1, -1) NGOÀI biên, bỏ qua
  hướng ( 1, 1) -> ô (1, 1) trong lưới, giá trị 7, tổng = 7
  kết quả = 7


Bảng trace `adjacentSum(4)` trên lưới `3×3` của Ví dụ 1 (`4` ở `(1, 1)`):

| hướng `(di, dj)` | ô | trong lưới? | giá trị | tổng |
|---|---|---|---|---|
| `(-1, 0)` | `(0, 1)` | có | 1 | 1 |
| `(1, 0)` | `(2, 1)` | có | 7 | 8 |
| `(0, -1)` | `(1, 0)` | có | 3 | 11 |
| `(0, 1)` | `(1, 2)` | có | 5 | 16 |

Kết quả `16` ✓. Với giá trị `1` ở góc trên-trái của lưới `4×4` (Ví dụ 2, `grid2[0][0] = 1`), `diagonalSum(1)`: ba hướng chéo ra ngoài biên, chỉ còn `(1, 1) = 7` ⇒ `7` (tự dựng, đã đối chiếu bằng vét cạn).

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (`dict` giá trị → vị trí + `cal` dùng chung): bám sát skeleton của LeetCode, ngắn gọn, mỗi truy vấn `O(1)`. Trong LeetCode, `List` đã được import sẵn; dòng `from typing import List` ở đây chỉ để chạy được trong notebook.

In [18]:
from typing import List


class NeighborSum:

    def __init__(self, grid: List[List[int]]):
        self.grid = grid
        self.d = {}                                    # giá trị -> (hàng, cột)
        for i, row in enumerate(grid):
            for j, x in enumerate(row):
                self.d[x] = (i, j)

    def adjacentSum(self, value: int) -> int:
        return self.cal(value, ((-1, 0), (1, 0), (0, -1), (0, 1)))        # trên, dưới, trái, phải

    def diagonalSum(self, value: int) -> int:
        return self.cal(value, ((-1, -1), (-1, 1), (1, -1), (1, 1)))      # bốn góc chéo

    def cal(self, value: int, dirs) -> int:
        i, j = self.d[value]
        n = len(self.grid)
        s = 0
        for di, dj in dirs:
            x, y = i + di, j + dj
            if 0 <= x < n and 0 <= y < n:              # chỉ cộng ô nằm trong lưới
                s += self.grid[x][y]
        return s


# Your NeighborSum object will be instantiated and called as such:
# obj = NeighborSum(grid)
# param_1 = obj.adjacentSum(value)
# param_2 = obj.diagonalSum(value)

print(run_ops(NeighborSum, ["NeighborSum", "adjacentSum", "adjacentSum", "diagonalSum", "diagonalSum"], [[grid1], [1], [4], [4], [8]]))   # [None, 6, 16, 16, 4]
print(run_ops(NeighborSum, ["NeighborSum", "adjacentSum", "diagonalSum"], [[grid2], [15], [9]]))                                            # [None, 23, 45]

[None, 6, 16, 16, 4]
[None, 23, 45]


## 10. Vì sao thuật toán đúng

- **Mỗi giá trị xác định đúng một ô.** Các phần tử đôi một khác nhau nên `pos[value]` là **duy nhất**; không có ghi đè sai khi dựng `dict`. Và đề đảm bảo `value` thuộc `[0, n² − 1]`; vì lưới có `n²` ô đôi một khác nhau trong đúng khoảng đó nên **mọi** `value` đều có mặt trong lưới (nên `pos[value]` không gây `KeyError`).
- **Láng giềng đúng định nghĩa.** Ô kề cạnh của `(i, j)` là `(i ± 1, j)`, `(i, j ± 1)`; ô chéo là `(i ± 1, j ± 1)`. Hai bảng hướng liệt kê đủ và không trùng các ô đó (4 + 4, không giao nhau).
- **Biên.** Ô láng giềng chỉ tồn tại khi `0 <= x < n` và `0 <= y < n`; kiểm tra này loại ô ngoài lưới (ô ở góc có 2 láng giềng kề cạnh và 1 láng giềng chéo, ô ở cạnh có 3 và 2, ô ở trong có 4 và 4). Cách 4 thay kiểm tra bằng viền `0` — cộng `0` không đổi tổng.
- **Trạng thái không đổi.** Lưới không bị sửa sau `__init__`, nên `pos` (hoặc các tổng tính sẵn) luôn khớp với lưới — kết quả không phụ thuộc thứ tự các lời gọi, và gọi lặp lại cùng một truy vấn cho cùng kết quả.
- **Đối chiếu độc lập.** Cách 1 (quét cả lưới) được so với bốn cách còn lại trên hàng nghìn lưới ngẫu nhiên và mọi giá trị ở mục 12.

## 11. Time / Space Complexity

Gọi `n` là cạnh lưới, `N = n²` là số ô, `Q` là số truy vấn (`Q <= 2N`).

| Cách | `__init__` | Mỗi truy vấn | Space |
|---|---|---|---|
| 1. Quét cả lưới mỗi lần | `O(1)` | `O(N)` | `O(1)` thêm |
| 2. `dict` giá trị → vị trí | `O(N)` | `O(1)` | `O(N)` |
| 3. Tính sẵn mọi tổng | `O(N)` | `O(1)` (tra mảng) | `O(N)` |
| 4. Viền `0` + mảng vị trí | `O(N)` | `O(1)` (không `if`) | `O(N)` |
| 5. Generator láng giềng | `O(N)` | `O(1)` | `O(N)` |

Với `n <= 10` (`N <= 100`) mọi cách đều rất nhanh; bài học thiết kế là **chuyển việc lặp lại sang bước khởi tạo**.

## 12. Bộ test

Gồm: ví dụ đề (qua `run_ops`), biên (góc, cạnh, ô giữa, `n = 3` nhỏ nhất, `n = 10` lớn nhất), phản ví dụ cho các lỗi thường gặp, đối chiếu các cách với nhau và với bản vét cạn độc lập trên mọi giá trị, ngẫu nhiên có seed đúng ràng buộc.

In [19]:
import random

classes = {
    "scan": NeighborSumScan,
    "dict": NeighborSumDict,
    "precompute": NeighborSumPrecompute,
    "padded": NeighborSumPadded,
    "generator": NeighborSumGenerator,
    "submit": NeighborSum,
}

# 1) Ví dụ của đề, chạy bằng run_ops giống LeetCode
ops1 = ["NeighborSum", "adjacentSum", "adjacentSum", "diagonalSum", "diagonalSum"]
args1 = [[grid1], [1], [4], [4], [8]]
ops2 = ["NeighborSum", "adjacentSum", "diagonalSum"]
args2 = [[grid2], [15], [9]]
for name, cls in classes.items():
    assert run_ops(cls, ops1, args1) == [None, 6, 16, 16, 4], name
    assert run_ops(cls, ops2, args2) == [None, 23, 45], name
print("PASS ví dụ đề")

# 2) Bản vét cạn độc lập (không dùng hướng hay dict) để làm "đáp án chuẩn"
def oracle(grid, value, kind):
    n = len(grid)
    for i in range(n):
        for j in range(n):
            if grid[i][j] == value:
                total = 0
                for x in range(n):
                    for y in range(n):
                        d_row, d_col = abs(x - i), abs(y - j)
                        if kind == "adj" and d_row + d_col == 1:      # cách 1 bước theo hàng HOẶC cột
                            total += grid[x][y]
                        if kind == "diag" and d_row == 1 and d_col == 1:
                            total += grid[x][y]
                return total

# 3) Biên và vị trí đặc biệt (tự dựng): góc, cạnh, giữa
g3 = [[3, 0, 7], [1, 8, 2], [5, 6, 4]]
checks = [("adjacentSum", 3, 1), ("diagonalSum", 3, 8),        # góc trên-trái
          ("adjacentSum", 0, 3 + 7 + 8), ("diagonalSum", 0, 1 + 2),      # cạnh trên
          ("adjacentSum", 8, 0 + 6 + 1 + 2), ("diagonalSum", 8, 3 + 7 + 5 + 4),   # ô giữa
          ("adjacentSum", 4, 2 + 6), ("diagonalSum", 4, 8)]    # góc dưới-phải
for name, cls in classes.items():
    obj = cls(g3)
    for method, value, expected in checks:
        assert getattr(obj, method)(value) == expected, (name, method, value)
for method, value, expected in checks:
    kind = "adj" if method == "adjacentSum" else "diag"
    assert oracle(g3, value, kind) == expected
print("PASS biên (góc, cạnh, giữa)")

PASS ví dụ đề
PASS biên (góc, cạnh, giữa)


In [20]:
# 4) Phản ví dụ: các cách làm sai (xem mục 13)
class SaiKhongKiemTraBien:
    def __init__(self, grid):
        self.grid = grid
        self.pos = {x: (i, j) for i, row in enumerate(grid) for j, x in enumerate(row)}

    def adjacentSum(self, value):
        i, j = self.pos[value]
        g = self.grid
        n = len(g)
        # SAI: chỉ chặn phía lớn (x < n), không chặn chỉ số âm -> g[-1] là hàng cuối
        total = 0
        for di, dj in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            x, y = i + di, j + dj
            if x < n and y < n:
                total += g[x][y]
        return total


sai = SaiKhongKiemTraBien(grid1)
print("sai adjacentSum(1):", sai.adjacentSum(1), "| đúng:", NeighborSum(grid1).adjacentSum(1))      # 13 vs 6
assert sai.adjacentSum(1) != NeighborSum(grid1).adjacentSum(1)

# 5) Đối chiếu mọi cách với oracle trên MỌI giá trị của lưới ví dụ và lưới 4x4 + lưới 10x10 cố định
for grid in (grid1, grid2, g3):
    n = len(grid)
    for name, cls in classes.items():
        obj = cls(grid)
        for v in range(n * n):
            assert obj.adjacentSum(v) == oracle(grid, v, "adj"), (name, v)
            assert obj.diagonalSum(v) == oracle(grid, v, "diag"), (name, v)
print("PASS mọi giá trị trên 3 lưới cố định, 6 lớp")

sai adjacentSum(1): 13 | đúng: 6
PASS mọi giá trị trên 3 lưới cố định, 6 lớp


In [21]:
# 6) Ngẫu nhiên có seed: n từ 3 đến 10, lưới là hoán vị của 0..n^2-1, số truy vấn tối đa 2*n^2
random.seed(3242)
total_queries = 0
for _ in range(300):
    n = random.randint(3, 10)
    values = list(range(n * n))
    random.shuffle(values)
    grid = [values[i * n:(i + 1) * n] for i in range(n)]
    objs = {name: cls(grid) for name, cls in classes.items()}
    for _ in range(random.randint(1, 2 * n * n)):
        v = random.randint(0, n * n - 1)
        kind = random.choice(["adj", "diag"])
        method = "adjacentSum" if kind == "adj" else "diagonalSum"
        ref = oracle(grid, v, kind)
        for name, obj in objs.items():
            assert getattr(obj, method)(v) == ref, (name, n, v, kind)
        total_queries += 1
print("PASS ngẫu nhiên: 300 lưới,", total_queries, "truy vấn, mọi lớp khớp bản vét cạn")

# 7) Ràng buộc: lưới n x n là hoán vị và 2*n^2 truy vấn trên lưới 10x10 chạy nhanh
import time
n = 10
values = list(range(n * n))
random.shuffle(values)
big = [values[i * n:(i + 1) * n] for i in range(n)]
t0 = time.perf_counter()
obj = NeighborSum(big)
for k in range(2 * n * n):
    obj.adjacentSum(k % (n * n))
    obj.diagonalSum(k % (n * n))
assert time.perf_counter() - t0 < 1.0
print("PASS 10x10 với 200 truy vấn (dưới 1 giây)")

PASS ngẫu nhiên: 300 lưới, 15051 truy vấn, mọi lớp khớp bản vét cạn
PASS 10x10 với 200 truy vấn (dưới 1 giây)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên `self.` — dữ liệu biến mất sau `__init__`

In [22]:
class SaiQuenSelf:
    def __init__(self, grid):
        pos = {}                                         # SAI: biến cục bộ, không phải self.pos
        for i, row in enumerate(grid):
            for j, x in enumerate(row):
                pos[x] = (i, j)

    def adjacentSum(self, value):
        i, j = self.pos[value]                           # AttributeError: không có self.pos
        return 0


try:
    SaiQuenSelf(grid1).adjacentSum(1)
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'SaiQuenSelf' object has no attribute 'pos'


**Vì sao sai:** `pos` không có `self.` chỉ tồn tại trong `__init__` rồi **biến mất**; các phương thức khác không thấy nó. **Cách sửa:** `self.pos = {}` (và `self.grid = grid` nếu cần dùng lưới sau).

### Lỗi 2: Quên kiểm tra biên — chỉ số âm quay vòng, sai thầm lặng

In [23]:
class SaiQuenBienAm:
    def __init__(self, grid):
        self.grid = grid
        self.pos = {x: (i, j) for i, row in enumerate(grid) for j, x in enumerate(row)}

    def adjacentSum(self, value):
        i, j = self.pos[value]
        n = len(self.grid)
        total = 0
        for di, dj in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            x, y = i + di, j + dj
            if x < n and y < n:                         # SAI: chỉ chặn phía lớn, quên x >= 0 và y >= 0
                total += self.grid[x][y]
        return total


print("sai adjacentSum(1):", SaiQuenBienAm(grid1).adjacentSum(1), "| đúng:", NeighborSum(grid1).adjacentSum(1))
print("sai adjacentSum(0):", SaiQuenBienAm(grid1).adjacentSum(0), "| đúng:", NeighborSum(grid1).adjacentSum(0))
assert SaiQuenBienAm(grid1).adjacentSum(1) != 6

sai adjacentSum(1): 13 | đúng: 6
sai adjacentSum(0): 12 | đúng: 4


**Vì sao sai:** với `1` ở hàng `0`, ô "phía trên" có `x = -1`; Python hiểu `grid[-1]` là **hàng cuối** (`[6, 7, 8]`) nên cộng thêm `7` (và với cột `0` thì cộng `grid[...][-1]`, tức cột cuối) mà **không báo lỗi** — kết quả `13` thay vì `6`. **Cách sửa:** kiểm tra đủ hai phía `0 <= x < n and 0 <= y < n`.

### Lỗi 3: Nhầm giá trị với chỉ số hàng

In [24]:
class SaiNhamGiaTri:
    def __init__(self, grid):
        self.grid = grid

    def adjacentSum(self, value):
        n = len(self.grid)
        i, j = value // n, value % n                    # SAI: giả định ô (i, j) luôn chứa giá trị i * n + j
        total = 0
        for di, dj in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            x, y = i + di, j + dj
            if 0 <= x < n and 0 <= y < n:
                total += self.grid[x][y]
        return total


print("lưới có thứ tự sẵn grid1:", SaiNhamGiaTri(grid1).adjacentSum(4), "| đúng:", NeighborSum(grid1).adjacentSum(4))     # trùng do may mắn
print("lưới ví dụ 2:", SaiNhamGiaTri(grid2).adjacentSum(15), "| đúng:", NeighborSum(grid2).adjacentSum(15))
assert SaiNhamGiaTri(grid1).adjacentSum(4) == 16 and SaiNhamGiaTri(grid2).adjacentSum(15) != 23

lưới có thứ tự sẵn grid1: 16 | đúng: 16
lưới ví dụ 2: 25 | đúng: 23


**Vì sao sai:** `value` là **số ghi trong ô**, không cho biết ô nằm ở đâu — chỉ trong lưới mẫu `0..8` xếp sẵn thứ tự mới có `giá_trị = i * n + j`, nên code sai vẫn ra đúng ở Ví dụ 1 nhưng sai ở Ví dụ 2 (lưới bị xáo trộn). Phải tìm tọa độ của `value` (quét hoặc dùng `dict`) rồi mới đi tới láng giềng. **Cách sửa:** `i, j = self.pos[value]`.

### Lỗi 4: Nhầm bộ hướng (kề cạnh ↔ kề chéo) hoặc lấy cả 8 hướng

In [25]:
ALL8 = ((-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1))


class SaiDuHuong:
    def __init__(self, grid):
        self.grid = grid
        self.pos = {x: (i, j) for i, row in enumerate(grid) for j, x in enumerate(row)}

    def _sum(self, value, dirs):
        i, j = self.pos[value]
        n = len(self.grid)
        return sum(self.grid[i + a][j + b] for a, b in dirs if 0 <= i + a < n and 0 <= j + b < n)

    def adjacentSum(self, value):
        return self._sum(value, ALL8)                   # SAI: lấy cả 8 ô, kể cả ô chéo

    def diagonalSum(self, value):
        return self._sum(value, ((-1, 0), (1, 0), (0, -1), (0, 1)))   # SAI: lấy bộ kề cạnh


sai = SaiDuHuong(grid1)
print("sai adjacentSum(4):", sai.adjacentSum(4), "| đúng:", NeighborSum(grid1).adjacentSum(4))     # 32 vs 16
print("sai diagonalSum(4):", sai.diagonalSum(4), "| đúng:", NeighborSum(grid1).diagonalSum(4))     # 16 vs 16 (trùng!)
print("sai diagonalSum(1):", sai.diagonalSum(1), "| đúng:", NeighborSum(grid1).diagonalSum(1))     # 6 vs 8
assert sai.adjacentSum(4) != 16 and sai.diagonalSum(1) != NeighborSum(grid1).diagonalSum(1)

sai adjacentSum(4): 32 | đúng: 16
sai diagonalSum(4): 16 | đúng: 16
sai diagonalSum(1): 6 | đúng: 8


**Vì sao sai:** "kề cạnh" là `4` ô (trên, dưới, trái, phải), "kề chéo" là `4` ô (bốn góc), **không giao nhau**; lấy `8` hướng cho `adjacentSum` cộng thừa các ô chéo. Lưu ý trường hợp `diagonalSum(4)` ở ví dụ đề lại **trùng** đáp án (`16`) dù dùng bộ hướng sai — một lý do nữa để test nhiều giá trị chứ không chỉ ví dụ mẫu. **Cách sửa:** hai bảng hướng riêng, mỗi bảng đúng `4` cặp.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Thiết kế lớp, nhiều truy vấn theo giá trị**" ⇒ hỏi: *việc gì lặp lại giữa các truy vấn?* Chuyển nó vào `__init__` (ở đây: tìm vị trí của giá trị).
- "**Giá trị đôi một khác nhau**" ⇒ giá trị là **khóa** hợp lệ cho `dict` (giá trị → vị trí), hoặc chỉ số mảng nếu là hoán vị `0..N−1`.
- Bài lưới: dùng **bảng độ lệch hướng** `(di, dj)` + một hàm kiểm tra biên; hai loại láng giềng chỉ khác bảng hướng ⇒ **dùng chung một hàm**.
- Python: **chỉ số âm không báo lỗi** — luôn kiểm tra `0 <= x < n` thay vì chỉ `x < n`.
- Với bảng nhỏ, có thể **tính sẵn mọi đáp án** (Cách 3) hoặc **thêm viền** để bỏ kiểm tra biên (Cách 4).
- Tự mô phỏng LeetCode bằng `run_ops` (hoặc `getattr`) để chạy dãy lệnh mẫu của đề.

## 15. Biến thể của bài

1. **Cả 8 láng giềng** (kề cạnh + kề chéo) — `eight_neighbor_sum`.
2. **Lưới `m × n` không vuông**, giá trị có thể **lặp** (khi đó `value → danh sách vị trí` và phải quyết định ô nào).
3. **Cập nhật giá trị**: thêm `update(value, new_value)` — vị trí giữ nguyên, nhưng `pos` (và các tổng tính sẵn nếu dùng Cách 3) phải được sửa.
4. **Bán kính `k`**: tổng các ô cách ô gốc tối đa `k` ô theo khoảng cách Manhattan hoặc Chebyshev.
5. **Liệt kê láng giềng** (danh sách giá trị) thay vì tổng; hoặc **giá trị lớn nhất** trong láng giềng.
6. Bài liên quan: **Game of Life (289)**, **Image Smoother (661)**, **Matrix Block Sum (1314)**, **Range Sum Query 2D – Immutable (304)**, **Design Underground System (1396)**.

In [26]:
class NeighborSumExtended:
    """Biến thể: 8 láng giềng, liệt kê láng giềng, bán kính Chebyshev k, và update(value, new_value)."""
    ADJ = ((-1, 0), (1, 0), (0, -1), (0, 1))
    DIAG = ((-1, -1), (-1, 1), (1, -1), (1, 1))

    def __init__(self, grid):
        self.grid = [row[:] for row in grid]               # sao chép để update không đổi lưới gốc
        self.pos = {x: (i, j) for i, row in enumerate(self.grid) for j, x in enumerate(row)}

    def neighbors(self, value, kind):
        i, j = self.pos[value]
        n = len(self.grid)
        dirs = self.ADJ if kind == "adj" else self.DIAG if kind == "diag" else self.ADJ + self.DIAG
        return sorted(self.grid[i + a][j + b] for a, b in dirs if 0 <= i + a < n and 0 <= j + b < n)

    def eight_neighbor_sum(self, value):
        return sum(self.neighbors(value, "all"))

    def radius_sum(self, value, k):
        i, j = self.pos[value]
        n = len(self.grid)
        return sum(self.grid[x][y] for x in range(max(0, i - k), min(n, i + k + 1))
                   for y in range(max(0, j - k), min(n, j + k + 1)) if (x, y) != (i, j))

    def update(self, value, new_value):
        i, j = self.pos.pop(value)
        self.grid[i][j] = new_value
        self.pos[new_value] = (i, j)


ext = NeighborSumExtended(grid1)
print(ext.neighbors(4, "adj"), ext.neighbors(4, "diag"))      # [1, 3, 5, 7] [0, 2, 6, 8]
print(ext.eight_neighbor_sum(4))                              # 32
print(ext.radius_sum(0, 1), ext.radius_sum(4, 2))             # 8 (1+3+4) ; 36 - 4 = 32
ext.update(4, 40)
print(ext.neighbors(1, "adj"))                                # [0, 2, 40]
assert ext.eight_neighbor_sum(40) == 32 and ext.radius_sum(0, 1) == 1 + 3 + 40
assert NeighborSumExtended(grid1).eight_neighbor_sum(0) == 1 + 3 + 4
assert NeighborSumExtended(grid1).radius_sum(4, 2) == 36 - 4

[1, 3, 5, 7] [0, 2, 6, 8]
32
8 32
[0, 2, 40]


## 16. So sánh các cách làm

| Cách | Ý tưởng | `__init__` | Mỗi truy vấn | Space | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Quét cả lưới | tìm ô mỗi lần hỏi | `O(1)` | `O(N)` | `O(1)` | baseline, hiểu đề |
| 2. `dict` + `cal` dùng chung | `pos[value]` rồi cộng hướng | `O(N)` | `O(1)` | `O(N)` | **nên nộp** |
| 3. Tính sẵn mọi tổng | `adj[v]`, `diag[v]` ở `__init__` | `O(N)` | `O(1)` tra mảng | `O(N)` | rất nhiều truy vấn |
| 4. Viền `0` | không cần kiểm tra biên | `O(N)` | `O(1)` | `O(N)` | ít `if`, nhớ lệch `+1` |
| 5. Generator láng giềng | `yield` từng láng giềng hợp lệ | `O(N)` | `O(1)` | `O(N)` | dễ mở rộng biến thể |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Với `grid = [[0, 1, 2], [3, 4, 5], [6, 7, 8]]` và `obj = NeighborSum(grid)`, không chạy code, đoán: `obj.adjacentSum(0)`, `obj.diagonalSum(0)`, `obj.adjacentSum(8)`, `obj.diagonalSum(2)`, và `{x: (i, j) for i, r in enumerate(grid) for j, x in enumerate(r)}[7]`.

**Bài 2 (tính tay).** Với `grid = [[3, 0, 7], [1, 8, 2], [5, 6, 4]]`: tìm vị trí của `8`, liệt kê các ô kề cạnh và kề chéo rồi tính `adjacentSum(8)` và `diagonalSum(8)`. Thử thêm `adjacentSum(3)` và `diagonalSum(3)` (ô ở góc).

**Bài 3 (điền TODO).** Hoàn thành lớp `NeighborSumTodo` theo Cách 2 (`dict` + `cal`).

In [27]:
class NeighborSumTodo:
    def __init__(self, grid):
        # TODO 1: lưu self.grid = grid
        # TODO 2: dựng self.pos: giá trị -> (hàng, cột) bằng enumerate hai tầng
        pass

    def _cal(self, value, dirs):
        # TODO 3: i, j = self.pos[value]; duyệt (di, dj) trong dirs; nếu 0 <= x < n và 0 <= y < n thì cộng self.grid[x][y]
        pass

    def adjacentSum(self, value):
        # TODO 4: gọi self._cal với bộ hướng kề cạnh
        pass

    def diagonalSum(self, value):
        # TODO 5: gọi self._cal với bộ hướng kề chéo
        pass

# KHÔNG tạo đối tượng NeighborSumTodo ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết lớp `NeighborSumPre` theo Cách 3 (**tính sẵn**): trong `__init__` tính `self.adj` và `self.diag` (dict `giá_trị → tổng`) cho **mọi** ô; `adjacentSum` / `diagonalSum` chỉ tra dict.

**Bài 5 (biến thể / mini challenge).** (a) Viết lớp `NeighborSum8` có thêm `allSum(value)` (tổng cả 8 láng giềng). (b) Viết `maxAdjacent(value)` trả về **giá trị lớn nhất** trong các láng giềng kề cạnh. (c) Viết hàm `corner_cells(grid)` trả về danh sách các giá trị ở **bốn góc** của lưới (từ trên-trái theo chiều kim đồng hồ) và dùng nó kiểm tra: với ô ở góc, `adjacentSum` có đúng `2` số hạng và `diagonalSum` có đúng `1` số hạng.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `0` ở góc `(0, 0)`: kề cạnh là ô phải `1` và ô dưới `3`; kề chéo chỉ có `(1, 1) = 4`. `7` ở hàng `2`, cột `1`. Bài 3: trong `__init__` nhớ `self.`.
- **Mức 2 (vừa):** Bài 2: `8` ở `(1, 1)` (giữa lưới): kề cạnh `0, 6, 1, 2`; kề chéo `3, 7, 5, 4`. `3` ở `(0, 0)`: kề cạnh `0, 1`; kề chéo `8`. Bài 4: duyệt từng ô, cộng các ô hợp lệ vào `self.adj[v]`, `self.diag[v]`.
- **Mức 3 (sâu):** Bài 5(a): `ALL8` = `ADJ + DIAG`. 5(b): `max` của danh sách láng giềng kề cạnh (luôn có ít nhất `2` vì `n >= 3`). 5(c): góc là `grid[0][0]`, `grid[0][-1]`, `grid[-1][-1]`, `grid[-1][0]` (ở đây dùng chỉ số âm **có chủ ý**).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
grid = [[0, 1, 2], [3, 4, 5], [6, 7, 8]]
obj = NeighborSum(grid)
assert obj.adjacentSum(0) == 1 + 3 == 4
assert obj.diagonalSum(0) == 4
assert obj.adjacentSum(8) == 5 + 7 == 12
assert obj.diagonalSum(2) == 4
assert {x: (i, j) for i, r in enumerate(grid) for j, x in enumerate(r)}[7] == (2, 1)

# Bài 2
g = [[3, 0, 7], [1, 8, 2], [5, 6, 4]]
og = NeighborSum(g)
assert og.d[8] == (1, 1)
assert og.adjacentSum(8) == 0 + 6 + 1 + 2 == 9
assert og.diagonalSum(8) == 3 + 7 + 5 + 4 == 19
assert og.adjacentSum(3) == 0 + 1 == 1
assert og.diagonalSum(3) == 8


# Bài 3
class NeighborSumTodoAns:
    def __init__(self, grid):
        self.grid = grid
        self.pos = {}
        for i, row in enumerate(grid):
            for j, x in enumerate(row):
                self.pos[x] = (i, j)

    def _cal(self, value, dirs):
        i, j = self.pos[value]
        n = len(self.grid)
        s = 0
        for di, dj in dirs:
            x, y = i + di, j + dj
            if 0 <= x < n and 0 <= y < n:
                s += self.grid[x][y]
        return s

    def adjacentSum(self, value):
        return self._cal(value, ((-1, 0), (1, 0), (0, -1), (0, 1)))

    def diagonalSum(self, value):
        return self._cal(value, ((-1, -1), (-1, 1), (1, -1), (1, 1)))


for g_, queries, expected in examples:
    assert [getattr(NeighborSumTodoAns(g_), name)(v) for name, v in queries] == expected


# Bài 4
class NeighborSumPre:
    def __init__(self, grid):
        n = len(grid)
        self.adj, self.diag = {}, {}
        for i in range(n):
            for j in range(n):
                v = grid[i][j]
                a = d = 0
                for di, dj in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                    x, y = i + di, j + dj
                    if 0 <= x < n and 0 <= y < n:
                        a += grid[x][y]
                for di, dj in ((-1, -1), (-1, 1), (1, -1), (1, 1)):
                    x, y = i + di, j + dj
                    if 0 <= x < n and 0 <= y < n:
                        d += grid[x][y]
                self.adj[v], self.diag[v] = a, d

    def adjacentSum(self, value):
        return self.adj[value]

    def diagonalSum(self, value):
        return self.diag[value]


for g_, queries, expected in examples:
    assert [getattr(NeighborSumPre(g_), name)(v) for name, v in queries] == expected


# Bài 5
class NeighborSum8(NeighborSum):
    def allSum(self, value):
        return self.adjacentSum(value) + self.diagonalSum(value)

    def maxAdjacent(self, value):
        i, j = self.d[value]
        n = len(self.grid)
        return max(self.grid[i + a][j + b] for a, b in ((-1, 0), (1, 0), (0, -1), (0, 1)) if 0 <= i + a < n and 0 <= j + b < n)


def corner_cells(grid):
    return [grid[0][0], grid[0][-1], grid[-1][-1], grid[-1][0]]          # chỉ số âm dùng có chủ ý


o8 = NeighborSum8(grid1)
assert o8.allSum(4) == 32 and o8.allSum(0) == 1 + 3 + 4
assert o8.maxAdjacent(4) == 7 and o8.maxAdjacent(0) == 3
assert corner_cells(grid1) == [0, 2, 8, 6]
for g_ in (grid1, grid2):
    n_ = len(g_)
    for corner in corner_cells(g_):
        i_, j_ = {x: (i, j) for i, r in enumerate(g_) for j, x in enumerate(r)}[corner]
        cnt_adj = sum(1 for a, b in ((-1, 0), (1, 0), (0, -1), (0, 1)) if 0 <= i_ + a < n_ and 0 <= j_ + b < n_)
        cnt_diag = sum(1 for a, b in ((-1, -1), (-1, 1), (1, -1), (1, 1)) if 0 <= i_ + a < n_ and 0 <= j_ + b < n_)
        assert (cnt_adj, cnt_diag) == (2, 1)
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đánh chỉ mục giá trị → vị trí ở `__init__`; hai loại tổng dùng chung một hàm với bảng hướng khác nhau)

```python
class NeighborSum:

    def __init__(self, grid: List[List[int]]):
        self.grid = grid
        self.d = {x: (i, j) for i, row in enumerate(grid) for j, x in enumerate(row)}

    def adjacentSum(self, value: int) -> int:
        return self.cal(value, ((-1, 0), (1, 0), (0, -1), (0, 1)))

    def diagonalSum(self, value: int) -> int:
        return self.cal(value, ((-1, -1), (-1, 1), (1, -1), (1, 1)))

    def cal(self, value: int, dirs) -> int:
        i, j = self.d[value]
        n = len(self.grid)
        return sum(self.grid[i + a][j + b] for a, b in dirs if 0 <= i + a < n and 0 <= j + b < n)
```

**Checklist**

- [ ] Mọi dữ liệu cần dùng lại đều lưu bằng `self.` trong `__init__` (`self.grid`, `self.d`)?
- [ ] Có kiểm tra biên **hai phía** `0 <= x < n and 0 <= y < n` (chỉ số âm của Python không báo lỗi)?
- [ ] Hai bảng hướng riêng, mỗi bảng đúng `4` cặp; không lẫn kề cạnh với kề chéo?
- [ ] Tìm vị trí của `value` bằng `dict` (không dùng `value` làm chỉ số hàng)?
- [ ] Đã thử ví dụ đề qua dãy lệnh (`[None, 6, 16, 16, 4]` và `[None, 23, 45]`) và các ô ở góc/cạnh?
- [ ] Skeleton đúng `class NeighborSum`, `__init__(self, grid: List[List[int]])`, `adjacentSum(self, value: int) -> int`, `diagonalSum(self, value: int) -> int`?